In [ ]:
# --- setup -------------------------------------------------------------------
from google.colab import drive
drive.mount('/content/drive')

REPO = '/content/pxr-repo'
!git clone -q https://github.com/pridem755/patient-or-xray.git $REPO 2>/dev/null || (cd $REPO && git pull -q)
%pip install -q -e $REPO

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

from pxr.config import load_config
from pxr.stats.power import (
    add_age_group,
    apply_coarsening_ladder,
    assign_tiers,
    cell_positive_counts,
    evaluate_cells,
    minimum_detectable_effect,
)

cfg = load_config(f'{REPO}/config/study_config.yaml')
ROOT = Path(cfg.paths['drive_root'])
COHORTS = ROOT / cfg.paths['cohorts']
OUT = ROOT / cfg.paths['analysis']
OUT.mkdir(parents=True, exist_ok=True)

power_cfg = cfg.analysis['power']
BASELINE = power_cfg['assumed_baseline_fnr']
MEANINGFUL = power_cfg['meaningful_effect']
REPLICATES = power_cfg['simulation_replicates']
LADDER = power_cfg['coarsening_ladder']
SENS_GRID = power_cfg['baseline_sensitivity_grid']
AGE_CUT = cfg.primary_age_threshold

print('config_hash :', cfg.config_hash)
print('assumed baseline :', BASELINE, '(FNR level at which the gap is detected)')
print('meaningful gap :', MEANINGFUL)
print('age contrast : <', AGE_CUT, ' vs >=', AGE_CUT, ' (fixed clinical cut-point)')
print('inferential sites :', cfg.inferential_sites)
print('simulation reps :', f'{REPLICATES:,}')

In [ ]:
cohorts = {}
for site in cfg.site_names:
    path = COHORTS / cfg.artifact_name('cohort', site=site)
    cohorts[site] = pd.read_parquet(path)
    print(f'{site:<10} {len(cohorts[site]):>7,} patients   {path.name}')

assert all(set(df.config_hash) == {cfg.config_hash} for df in cohorts.values()), \
    'cohort config_hash does not match the current config - rebuild in notebook 02'

In [ ]:
# --- power analysis -----------------------------------------------------------
rng = np.random.default_rng(0)
rows = []
for n in (50, 100, 250, 500, 1000, 2500, 5000):
    mde = minimum_detectable_effect(n, n, baseline=BASELINE, replicates=1500, rng=rng)
    rows.append({'positives per group': f'{n:,}',
                 'smallest detectable gap': f'{mde * 100:.1f} pp'})
print(pd.DataFrame(rows).to_string(index=False))
print(f'\nA {MEANINGFUL:.0%} gap needs roughly 400+ positives in each group.')

In [ ]:
# --- age/sex coupling ------------------------------------------------
coupling = []
for site, df in cohorts.items():
    d = df.assign(is_ap=(df.view == 'AP').astype(int), age_decade=df.age / 10)
    model = smf.logit('is_ap ~ age_decade + C(sex)', data=d).fit(disp=0)
    ci = model.conf_int()
    for term in model.params.index:
        if term == 'Intercept':
            continue
        coupling.append({
            'site': site,
            'term': term,
            'odds_ratio': np.exp(model.params[term]),
            'ci_low': np.exp(ci.loc[term, 0]),
            'ci_high': np.exp(ci.loc[term, 1]),
            'p_value': model.pvalues[term],
        })

coupling = pd.DataFrame(coupling)
print(coupling.round(4).to_string(index=False))
print('\nAn odds ratio above 1 for age_decade means older patients are imaged AP more often.')

In [ ]:
# observed P(AP) by band, alongside the model — the descriptive companion to §1
print('P(AP) by age band:')
print(pd.concat([
    df.assign(site=site).groupby(['site', 'age_bin'], observed=True)
      .view.apply(lambda v: (v == 'AP').mean())
    for site, df in cohorts.items()
]).unstack().round(3).to_string())

In [ ]:
# --- inferential cells ------------------------------------------------
counts = pd.concat(
    [cell_positive_counts(df, cfg.analysis_labels, site=site, age_threshold=AGE_CUT)
     for site, df in cohorts.items()],
    ignore_index=True,
)
print(f'{len(counts)} inferential cells (sex and age contrasts x AP/PA x label x site)\n')
print('smallest cells:')
print(counts.assign(smaller=counts[['n_a', 'n_b']].min(axis=1))
            .nsmallest(10, 'smaller')
            .loc[:, ['site', 'label', 'view', 'stratum', 'level_a', 'n_a', 'level_b', 'n_b']]
            .to_string(index=False))

In [ ]:
# --- power evaluation ------------------------------------------------
table = evaluate_cells(
    counts,
    baseline=BASELINE,
    alpha=power_cfg['alpha'],
    target_power=power_cfg['target_power'],
    replicates=REPLICATES,
    meaningful_effect=MEANINGFUL,
    seed=cfg.splits['seed'],
)

print(f'powered cells: {int(table.powered.sum())} / {len(table)}\n')
print('by site:'); print(table.groupby('site').powered.agg(['sum', 'count']).to_string())
print('\nby stratum:'); print(table.groupby('stratum').powered.agg(['sum', 'count']).to_string())

In [ ]:
# per site x label, is the comparison estimable?
pivot = (table.groupby(['label', 'site']).powered.all().unstack()
              .reindex(cfg.analysis_labels))
print('all cells powered (both views, both strata):')
print(pivot.to_string())

In [ ]:
# --- coarsening ladder ------------------------------------------------
final_table, applied = apply_coarsening_ladder(
    cohorts,
    cfg.analysis_labels,
    LADDER,
    age_threshold=AGE_CUT,
    baseline=BASELINE,
    alpha=power_cfg['alpha'],
    target_power=power_cfg['target_power'],
    replicates=REPLICATES,
    meaningful_effect=MEANINGFUL,
    seed=cfg.splits['seed'],
)

print('rung reached per label:')
for label, rung in applied.items():
    print(f'{label:<18} {rung}')

In [ ]:
# --- tier assignment -----------------------------------------------------------
tiers = assign_tiers(
    final_table,
    cfg.analysis_labels,
    secondary_lane=cfg.secondary_lane,
    inferential_sites=cfg.inferential_sites,
)
print(f'tiers gated on: {cfg.inferential_sites}')
print('(other sites are evaluated and reported, but do not decide the tier)\n')
print(tiers.round(3).to_string(index=False))

primary = tiers.loc[tiers.tier == 'primary', 'label'].tolist()
print(f'\nPRIMARY FAMILY ({len(primary)}): {primary}')
print(f'EXPLORATORY: {tiers.loc[tiers.tier == "exploratory", "label"].tolist()}')
print(f'DESCRIPTIVE: {tiers.loc[tiers.tier == "descriptive", "label"].tolist()}')

In [ ]:
# --- sensitivity analysis ------------------------------------------------
sensitivity = []
for baseline in SENS_GRID:
    alt = evaluate_cells(counts, baseline=baseline, alpha=power_cfg['alpha'],
                         target_power=power_cfg['target_power'],
                         replicates=max(2000, REPLICATES // 5),
                         meaningful_effect=MEANINGFUL, seed=cfg.splits['seed'])
    alt_tiers = assign_tiers(alt, cfg.analysis_labels,
                             secondary_lane=cfg.secondary_lane,
                             inferential_sites=cfg.inferential_sites)
    sensitivity.append(alt_tiers.set_index('label')['tier'].rename(f'FNR={baseline:.0%}'))

sens = pd.concat(sensitivity, axis=1).reindex(cfg.analysis_labels)
sens['stable'] = sens.nunique(axis=1) == 1
print(sens.to_string())


In [ ]:
# --- save results ------------------------------------------------
final_table.to_csv(OUT / f'cell_power_{cfg.config_hash}.csv', index=False)
tiers.to_csv(OUT / f'label_tiers_{cfg.config_hash}.csv', index=False)
coupling.to_csv(OUT / f'acquisition_coupling_{cfg.config_hash}.csv', index=False)
sens.to_csv(OUT / f'tier_sensitivity_{cfg.config_hash}.csv')
print('saved to', OUT)

In [ ]:
# --- integrity cell ------------------
print(f'config_hash  : {cfg.config_hash}')
print(f'baseline FNR : {BASELINE}   meaningful gap: {MEANINGFUL}')
print(f'simulation reps : {REPLICATES}   seed: {cfg.splits["seed"]}')
print(f'cells evaluated : {len(final_table)}')
print(f'cells powered : {int(final_table.powered.sum())}')
print(f'primary family : {primary}')
print(f'tier stable across baselines: {bool(sens.stable.all())}')